# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which, from the repository root, you can run this via:

```bash
jupyter nbconvert --to notebook --execute _scripts/talkmap.ipynb --output talkmap_out.ipynb --output-dir=_scripts --ExecutePreprocessor.cwd=.
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import os
import shutil
import re
import time
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

# Newer versions of nbconvert/nbclient start the kernel in the notebook's
# own directory (_scripts/) and ignore --ExecutePreprocessor.cwd=. All paths
# in this notebook are relative to the repository root, so normalise here.
if os.path.basename(os.getcwd()) == '_scripts':
    os.chdir('..')


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Nominatim's usage policy caps requests at 1/second
RATE_LIMIT_SECONDS = 1

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""


def geocode_with_fallback(query, timeout):
    """Geocode, falling back to a shorter (less specific) query if no match is found."""
    time.sleep(RATE_LIMIT_SECONDS)
    result = geocoder.geocode(query, timeout=timeout)
    if result is None and "," in query:
        return geocode_with_fallback(query.split(",", 1)[1].strip(), timeout)
    return result

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description: title, month & year, exact city, and the
    # conference/institute (venue covers both "it was a conference" and
    # "for invited talks, the institute")
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    city = data.get('city', location.split(',')[0]).strip()
    country = location.rsplit(',', 1)[-1].strip()
    month_year = data['date'].strftime('%b %Y')
    description = f"<strong>{title}</strong><br>{month_year}<br>{venue}<br>{city}, {country}"
    if data.get('online'):
        description += "<br><em>(online)</em>"

    # Strip trailing "(online)"-style annotations before geocoding
    geocode_query = re.sub(r"\s*\([^)]*\)\s*$", "", location)

    # Geocode the location and report the status
    try:
        result = geocode_with_fallback(geocode_query, TIMEOUT)
        if result is None:
            print(f"Warning: no geocode match found for {geocode_query}, skipping pin")
            continue
        location_dict[description] = result
        print(description, result)
    except ValueError as ex:
        print(f"Error: geocode failed on input {geocode_query} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {geocode_query} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {geocode_query} with message {ex}")

<strong>Monolithic convex limiting techniques for finite element methods applied to hyperbolic PDEs</strong><br>Feb 2024<br>School of Engineering Science, Lappeenranta-Lahti University of Technology<br>Lappeenranta, Finland Lappeenranta, Lappeenrannan seutukunta, Etelä-Karjala, Manner-Suomi, Suomi / Finland


<strong>Bound-preserving and entropy-stable algebraic flux correction schemes for the shallow water equations with topography</strong><br>Jun 2022<br>ECCOMAS Congress 2022: 8th European Congress on Computational Methods in Applied Sciences and Engineering<br>Oslo, Norway Oslo, Norge


<strong>Spin-up of a stratified ocean with topography</strong><br>Jun 2025<br>ICGAFD: International Conference on Geophysical and Astrophysical Fluid Dynamics<br>Plouzané, France Plouzané, Brest, Finistère, Bretagne, France métropolitaine, 29280, France


<strong>Algebraically stabilized enriched Galerkin methods for shallow water flow</strong><br>Jun 2021<br>SIAM GS: SIAM Conference on Mathematical and Computational Issues in the Geosciences<br>Milano, Italy<br><em>(online)</em> Milano, Rodano, Milano, Lombardia, Italia


<strong>Algebraic flux correction for high-order discontinuous Galerkin methods</strong><br>Sep 2021<br>Hirschegg workshop on conservation laws<br>Hirschegg, Austria Hirschegg, Mittelberg, Bezirk Bregenz, Vorarlberg, 6992, Österreich


...

<strong>FESTUNG: An introduction to the software and example application on the basis of the shallow-water equations</strong><br>Dec 2017<br>Mechanics Institute, Chinese Academy of Sciences<br>Beijing, China 北京市, 中国


<strong>Failsafe limiting for high-order WENO-stabilized continuous Galerkin discretizations of hyperbolic systems</strong><br>Jul 2025<br>MoST: Modeling and Simulation of Transport Phenomena<br>Treis-Karden, Germany Treis-Karden, Cochem, Landkreis Cochem-Zell, Rheinland-Pfalz, 56253, Deutschland


<strong>Matrix-free advection-based remap algorithms for Lagrangian/ALE methods</strong><br>Mar 2019<br>Center for Applied Scientific Computing, LLNL<br>Livermore, California Livermore, Alameda County, California, United States


<strong>Property-preserving discontinuous Galerkin methods for hyperbolic conservation laws</strong><br>Aug 2021<br>Banff International Research Station for Mathematical Innovation and Discovery: Bound-Preserving Space and Time Discretizations for Convection-Dominated Problems<br>Casa Matemática Oaxaca, Mexico<br><em>(online)</em> México


<strong>Modified shallow-water equations for direct bathymetry reconstruction</strong><br>Dec 2017<br>Numerical Methods for Shallow Water Equations and Related Models<br>Shenzhen, China 深圳市, 广东省, 中国


In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="_talkmap", hashed_usernames=False)

# getorg writes its own default map.html/screen.css (Mercator tiles,
# fixed 800x600 box) on every run; overwrite them with our customized
# versions (single non-repeating image basemap, capped zoom, no grey box).
shutil.copyfile("_scripts/talkmap_assets/map.html", "_talkmap/map.html")
shutil.copyfile("_scripts/talkmap_assets/screen.css", "_talkmap/leaflet_dist/screen.css")

'_talkmap/leaflet_dist/screen.css'